---
title: "Practice Activity 4.1"
author: Lily
format:
    html:
        embed-resources: true
        code-line-numbers: true
---
**GitHub Repository**: <https://github.com/lilysteinberg/GSB-544---Computing-and-ML/tree/main/Week%204>

# XML, HTML, and Web Scraping

JSON and XML are two different ways to represent hierarchical data. Which one is better? There are lots of articles online which discuss similarities and differences between JSON and XML and their advantages and disadvantages. Both formats are still in current usage, so it is good to be familiar with both. However, JSON is more common, so we'll focus on working with JSON representations of hierarchical data.

The reading covered an example of using Beautiful Soup to parse XML. Rather than doing another example XML now, we'll skip straight to scraping HTML from a webpage. Both HTML and XML can be parsed in a similar way with Beautiful Soup.

In [ ]:
import pandas as pd

## Scraping an HTML table with Beautiful Soup

Open the URL https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population and scroll down until you see a table of the cities in the U.S. with population over 100,000 (as of Jul 1, 2022). We'll use Beautiful Soup to scrape information from this table.

Read in the HTML from the URL using the `requests` library.

In [ ]:
import requests

URL = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(URL, headers=HEADERS)

Use Beautiful Soup to parse this string into a tree called `soup`

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

To find an HTML tag corresponding to a specific element on a webpage, right-click on it and choose "Inspect element". Go to the cities table Wikipedia page and do this now.

You should find that the cities table on the Wikipedia page corresponds to the element

```
<table class="wikitable sortable jquery-tablesorter" style="text-align:center">
```

There are many `<table>` tags on the page.

In [ ]:
len(soup.find_all("table"))

10

We can use attributes like `class=` and `style=` to narrow down the list.

In [ ]:
len(soup.find_all("table",
                  attrs={
                      #found by finding the HTML when inspecting
                      "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
                      "style": "text-align:right"}
                  ))

1

At this point, you can manually inspect the tables on the webpage to find that the one we want is the first one (see `[0]` below). We'll store this as `table`.

In [ ]:
table = soup.find_all("table",
                  attrs={
                      "class": "sortable wikitable sticky-header-multi static-row-numbers sort-under col1left col2center",
                      "style": "text-align:right"}
                  )[0]

In [ ]:
cities = table.find_all("tr")
len(cities)

348

**Now you will write code to scrape the information in `table` to create a Pandas data frame with one row for each city and columns for: city, state, population (2022 estimate), and 2020 land area (sq mi).** Refer to the Notes/suggestions below as you write your code. A few Hints are provided further down, but try coding first before looking at the hints.

Notes/suggestions:

- Use as a guide the code from the reading that produced the data frame of Statistics faculty
- Inspect the page source as you write your code
- You will need to write a loop to get the information for all cities, but you might want to try just scraping the info for New York first
- You will need to pull the text from the tag. If `.text` returns text with "\n" at the end, try `.get_text(strip = True)` instead of `.text`
- Don't forget to convert to a Pandas Data Frame; it should have 333 rows and 4 columns
- The goal of this exercise is just to create the Data Frame. If you were going to use it --- e.g., what is the population density for all cities in CA? --- then you would need to clean the data first (to clean strings and convert to quantitative). (You can use Beautiful Soup to do some of the cleaning for you, but that goes beyond our scope.)

In [ ]:
#do New York first

#extract info from new york row
new_york = table.find_all("tr")[2]
new_york

<tr>
<td style="background-color:#cfecec"><i><a href="/wiki/New_York_City" title="New York City">New York</a></i><sup class="reference" id="cite_ref-5"><a href="#cite_note-5"><span class="cite-bracket">[</span>c<span class="cite-bracket">]</span></a></sup>
</td>
<td><a href="/wiki/New_York_(state)" title="New York (state)">NY</a>
</td>
<td style="text-align:right;">8,478,072
</td>
<td style="text-align:right;">8,804,190
</td>
<td style="text-align:right;"><span data-sort-value="2999630000000000000♠" style="display:none"></span><span style="color:red">−3.70%</span>
</td>
<td style="text-align:right;">300.5
</td>
<td style="text-align:right;">778.3
</td>
<td style="text-align:right;">29,298
</td>
<td style="text-align:right;">11,312
</td>
<td><small><span class="geo-inline"><style data-mw-deduplicate="TemplateStyles:r1156832818">.mw-parser-output .geo-default,.mw-parser-output .geo-dms,.mw-parser-output .geo-dec{display:inline}.mw-parser-output .geo-nondefault,.mw-parser-output .geo-mult

In [ ]:
# extract info from new york row
cells = new_york.find_all("td")
cells

[<td style="background-color:#cfecec"><i><a href="/wiki/New_York_City" title="New York City">New York</a></i><sup class="reference" id="cite_ref-5"><a href="#cite_note-5"><span class="cite-bracket">[</span>c<span class="cite-bracket">]</span></a></sup>
 </td>,
 <td><a href="/wiki/New_York_(state)" title="New York (state)">NY</a>
 </td>,
 <td style="text-align:right;">8,478,072
 </td>,
 <td style="text-align:right;">8,804,190
 </td>,
 <td style="text-align:right;"><span data-sort-value="2999630000000000000♠" style="display:none"></span><span style="color:red">−3.70%</span>
 </td>,
 <td style="text-align:right;">300.5
 </td>,
 <td style="text-align:right;">778.3
 </td>,
 <td style="text-align:right;">29,298
 </td>,
 <td style="text-align:right;">11,312
 </td>,
 <td><small><span class="geo-inline"><style data-mw-deduplicate="TemplateStyles:r1156832818">.mw-parser-output .geo-default,.mw-parser-output .geo-dms,.mw-parser-output .geo-dec{display:inline}.mw-parser-output .geo-nondefault,.mw-

In [ ]:
#city
cells[0].find("a").text

'New York'

In [ ]:
#state
cells[1].find("a").text

'NY'

In [ ]:
#pop
cells[2].get_text(strip=True)

'8,478,072'

In [ ]:
#land mass
cells[5].get_text(strip=True)

'300.5'

For all cities, not just New York:

In [ ]:
# initialize an empty list
rows = []

# iterate over all rows in the table
for cities in table.find_all("tr")[2:]:

    # Get all the cells (<td>) in the row.
    cells = cities.find_all("td")

    # The information we need is the text between tags.

    # Find the the name of the city
    name_tag = cells[0].find("a") or cells[0]
    name = name_tag.text

    # Find the state
    state_tag = cells[1].find("a") or cells[1]
    state = state_tag.text

    # Find the 2024 estimate population
    pop_tag = cells[2] or cells[2]
    pop = pop_tag.get_text(strip=True)

    # Find the 2020 land area
    land_tag = cells[5] or cells[5]
    land = land_tag.get_text(strip=True)

    # Append this data
    rows.append({
        "city": name,
        "state": state,
        "2024 population estimate": pop,
        "2020 land area (sq mi)": land
    })

In [ ]:
pd.DataFrame(rows)

,city,state,2024 population estimate,2020 land area (sq mi)
0,New York,NY,"8,478,072",300.5
1,Los Angeles,CA,"3,878,704",469.5
2,Chicago,IL,"2,721,308",227.7
3,Houston,TX,"2,390,125",640.4
4,Phoenix,AZ,"1,673,164",518.0
...,...,...,...,...
341,Deltona,FL,"100,513",37.3
342,Federal Way,WA,"100,252",22.3
343,San Angelo,TX,"100,159",59.7
344,Tracy,CA,"100,136",25.9


Hints:

- Each city is a row in the table; find all the `<tr>` tags to find all the cities
- Look for the `<td>` tag to see table entries within a row
- The rank column is represented by `<th>` tags, rather than `<td>` tags. So within a row, the first (that is, `[0]`) `<td>` tag corresponds to the city name.

## Aside: Scraping an HTML table with Pandas



The Pandas command `read_html` can be used to scrape information from an HTML table on a webpage.

We can call `read_html` on the URL.

In [ ]:
#pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population")

URL = "https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population"
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(URL, headers=HEADERS)

However, this scrapes all the tables on the webpage, not just the one we want. As with Beautiful Soup, we can narrow the search by specifying the table attributes.

In [ ]:
#pd.read_html("https://en.wikipedia.org/wiki/List_of_United_States_cities_by_population", attrs = {'class': 'wikitable sortable', "style": "text-align:center"})

from io import StringIO
df = pd.read_html(StringIO(response.text), attrs = {"style": "text-align:right"})
df

[            City  ST 2024 estimate 2020 census  Change 2020 land area          \
             City  ST 2024 estimate 2020 census  Change            mi2     km2   
 0    New York[c]  NY       8478072     8804190  −3.70%          300.5   778.3   
 1    Los Angeles  CA       3878704     3898747  −0.51%          469.5  1216.0   
 2        Chicago  IL       2721308     2746388  −0.91%          227.7   589.7   
 3        Houston  TX       2390125     2304580  +3.71%          640.4  1658.6   
 4        Phoenix  AZ       1673164     1608139  +4.04%          518.0  1341.6   
 ..           ...  ..           ...         ...     ...            ...     ...   
 341      Deltona  FL        100513       93692  +7.28%           37.3    96.6   
 342  Federal Way  WA        100252      101030  −0.77%           22.3    57.8   
 343   San Angelo  TX        100159       99893  +0.27%           59.7   154.6   
 344        Tracy  CA        100136       93000  +7.67%           25.9    67.1   
 345      Sunris

This still returns 3 tables. As we remarked above, the table that we want is the first one (see `[0]` below).

In [ ]:
df2 = pd.read_html(StringIO(response.text), attrs = {"style": "text-align:right"})[0]
df2

City  ST 2024 estimate 2020 census  Change 2020 land area          \
            City  ST 2024 estimate 2020 census  Change            mi2     km2   
0    New York[c]  NY       8478072     8804190  −3.70%          300.5   778.3   
1    Los Angeles  CA       3878704     3898747  −0.51%          469.5  1216.0   
2        Chicago  IL       2721308     2746388  −0.91%          227.7   589.7   
3        Houston  TX       2390125     2304580  +3.71%          640.4  1658.6   
4        Phoenix  AZ       1673164     1608139  +4.04%          518.0  1341.6   
..           ...  ..           ...         ...     ...            ...     ...   
341      Deltona  FL        100513       93692  +7.28%           37.3    96.6   
342  Federal Way  WA        100252      101030  −0.77%           22.3    57.8   
343   San Angelo  TX        100159       99893  +0.27%           59.7   154.6   
344        Tracy  CA        100136       93000  +7.67%           25.9    67.1   
345      Sunrise  FL        100128       97335  +2.87%           16.2    42.0   

    2020 density                                      Location  
           / mi2  / km2                               Location  
0          29298  11312    40°40′N 73°56′W﻿ / ﻿40.66°N 73.94°W  
1           8304   3206  34°01′N 118°25′W﻿ / ﻿34.02°N 118.41°W  
2          12061   4657    41°50′N 87°41′W﻿ / ﻿41.84°N 87.68°W  
3           3599   1390    29°47′N 95°23′W﻿ / ﻿29.79°N 95.39°W  
4           3105   1199  33°34′N 112°05′W﻿ / ﻿33.57°N 112.09°W  
..           ...    ...                                    ...  
341         2512    970    28°55′N 81°13′W﻿ / ﻿28.91°N 81.21°W  
342         4530   1750  47°19′N 122°20′W﻿ / ﻿47.31°N 122.34°W  
343         1673    646  31°26′N 100°27′W﻿ / ﻿31.44°N 100.45°W  
344         3591   1386  37°44′N 121°27′W﻿ / ﻿37.73°N 121.45°W  
345         6008   2320    26°10′N 80°16′W﻿ / ﻿26.17°N 80.26°W  

[346 rows x 10 columns]

Wait, that seemed much easier than using Beautiful Soup, and it returned a data frame, and we even got for free some formatting like removing the commas from the population! Why didn't we just use `read_html` in the first place? It's true the `read_html` works well when scraping information from an HTML *table*. Unfortunately, you often want to scrape information from a webpage that isn't conveniently stored in an HTML table, in which case `read_html` won't work. (It only searches for `<table>`, `<th>`, `<tr>`, and `<td>` tags, but there are many other HTML tags.) Though Beautiful Soup is not as simple as `read_html`, it is more flexible and thus more widely applicable.

## Scraping information that is NOT in a `<table>` with Beautiful Soup

The Cal Poly course catalog http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory contains a list of courses offered by the Statistics department. **You will scrape this website to obtain a Pandas data frame with one row for each DATA or STAT course and two columns: course name and number (e.g, DATA 301. Introduction to Data Science) and term typically offered (e.g., Term Typically Offered: F, W, SP).**

Note: Pandas `read_html` is not help here since the courses are not stored in a `<table>.`

In [ ]:
pd.read_html("http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory")

[                                        Program name   Program type
 0                              Actuarial Preparation          Minor
 1  Cross Disciplinary Studies Minor in Bioinforma...          Minor
 2   Cross Disciplinary Studies Minor in Data Science          Minor
 3                                         Statistics  BS, MS, Minor]


Notes/suggestions:


- Inspect the page source as you write your code
- The courses are not stored in a `<table>`. How are they stored?
- You will need to write a loop to get the information for all courses, but you might want to try just scraping the info for DATA 100 first
- What kind of tag is the course name stored in? What is the `class` of the tag?
- What kind of tag is the quarter(s) the course is offered stored in? What is the `class` of the tag? Is this the only tag of this type with the class? How will you get the one you want?
- You don't have to remove the number of units (e.g., 4 units) from the course name and number, but you can try it if you want
- You will need to pull the text from the tag. If `.text` returns text with "\n" at the end, try `get_text(strip = True)` instead of `text`
- Don't forget to convert to a Pandas Data Frame; it should have 74 rows and 2 columns
- The goal of this exercise is just to create the Data Frame. If you were going to use it then you might need to clean the data first. (You can use Beautiful Soup to do some of the cleaning for you, but that goes beyond our scope.)



In [ ]:
# set up beautiful soup scrape
response = requests.get("http://catalog.calpoly.edu/collegesandprograms/collegeofsciencemathematics/statistics/#courseinventory")
soup = BeautifulSoup(response.text, "html.parser")

In [ ]:
# save scrape to table object
table = soup.find_all("div", {"class": "courseblock"})
# check how many coures on page
len(table)

74

In [ ]:
#visual check to confirm table object is the one I meant it to be
table

[<div class="courseblock">
 <p class="courseblocktitle"><strong>DATA 100. Data Science for All I.
 <span class="courseblockhours">4 units
 </span></strong></p><div class="noindent courseextendedwrap">
 <p class="noindent">Term Typically Offered: F, W, SP</p><p class="noindent">2020-21 or later catalog: GE Area B4</p><p class="noindent">2019-20 or earlier catalog: GE Area B4</p><p>Prerequisite: <a class="bubblelink code" href="/search/?P=MATH%20115" onclick="return showCourse(this, 'MATH 115');" title="MATH 115">MATH 115</a>, <a class="bubblelink code" href="/search/?P=MATH%20116" onclick="return showCourse(this, 'MATH 116');" title="MATH 116">MATH 116</a>, <a class="bubblelink code" href="/search/?P=MATH%20118" onclick="return showCourse(this, 'MATH 118');" title="MATH 118">MATH 118</a>, or Appropriate Math Placement Level.</p></div>
 <div class="courseblockdesc">
 <p>Basic approaches for answering questions using data.  Emphasis on working with tabular data in spreadsheet software to 

Working on first class first, before attempting the loop for all data

In [ ]:
# get data for first class block
courses = soup.find_all("div", {"class": "courseblock"})[0]
courses

<div class="courseblock">
<p class="courseblocktitle"><strong>DATA 100. Data Science for All I.
<span class="courseblockhours">4 units
</span></strong></p><div class="noindent courseextendedwrap">
<p class="noindent">Term Typically Offered: F, W, SP</p><p class="noindent">2020-21 or later catalog: GE Area B4</p><p class="noindent">2019-20 or earlier catalog: GE Area B4</p><p>Prerequisite: <a class="bubblelink code" href="/search/?P=MATH%20115" onclick="return showCourse(this, 'MATH 115');" title="MATH 115">MATH 115</a>, <a class="bubblelink code" href="/search/?P=MATH%20116" onclick="return showCourse(this, 'MATH 116');" title="MATH 116">MATH 116</a>, <a class="bubblelink code" href="/search/?P=MATH%20118" onclick="return showCourse(this, 'MATH 118');" title="MATH 118">MATH 118</a>, or Appropriate Math Placement Level.</p></div>
<div class="courseblockdesc">
<p>Basic approaches for answering questions using data.  Emphasis on working with tabular data in spreadsheet software to provide

In [ ]:
# get data for first class's title
courses.find("p", {"class": "courseblocktitle"}, "strong").get_text(strip = True).replace('\xa0', ' ').replace('\xad', '')

'DATA 100. Data Science for All I.4 units'

In [ ]:
# get data for first class's term
courses.find("p", {"class" : "noindent"}).get_text(strip = True)

'Term Typically Offered: F, W, SP'

Loop for all data

In [ ]:
# make the loop

# initialize an empty list
rows = []

# iterate over all rows
for courses in soup.find_all("div", {"class": "courseblock"})[0:]:

    # Find the the name of the course
    course_tag = courses.find("p", {"class": "courseblocktitle"}, "strong")
    course = course_tag.get_text(strip = True).replace('\xa0', ' ').replace('\xad', '')

    # Find the term of the course
    term_tag = courses.find("p", {"class" : "noindent"})
    term = term_tag.get_text(strip = True)

    # Append this data.
    rows.append({
        "course": course,
        "term": term
    })

In [ ]:
pd.DataFrame(rows)

,course,term
0,DATA 100. Data Science for All I.4 units,"Term Typically Offered: F, W, SP"
1,DATA 301. Introduction to Data Science.4 units,"Term Typically Offered: F, W, SP"
2,DATA 401. Data Science Process and Ethics.3 units,Term Typically Offered: F
3,DATA 402. Mathematical Foundations of Data Sci...,Term Typically Offered: F
4,DATA 403. Data Science Projects Laboratory.1 unit,Term Typically Offered: F
...,...,...
69,STAT 551. Statistical Learning with R.4 units,Term Typically Offered: F
70,STAT 566. Graduate Consulting Practicum.2 units,Term Typically Offered: SP
71,STAT 570. Selected Advanced Topics.1-4 units,Term Typically Offered: TBD
72,STAT 590. Graduate Seminar in Statistics.1 unit,"Term Typically Offered: F, W, SP"


Hints:

- Each course is represented by a `<div>` with `class=courseblock`, so you can find all the courses with `soup.find_all("div", {"class": "courseblock"})`
- The course name is in a `<p>` tag with `class=courseblocktitle`, inside a `<strong>` tag. (Though I don't think we need to find the strong tag here.)
- The term typically offered is in `<p>` tag with `class=noindent`. However, there are several tags with this class; term typically offered is the first one.
- If you want to use Beautiful Soup to remove the course units (e.g., 4 units), find the `<span>` tag within the course name tag and `.extract()` this span tag